# 03b -- Stage 3 (Kaggle): Train on the Full Merged Dataset

**Why this notebook exists**: Colab's free-tier GPU quota has been the
recurring bottleneck through this whole project -- Stage 3's retrain
ran out of GPU quota mid-epoch more than once, with nothing saved
(the training loop only checkpoints after a full epoch completes, and
only if `balanced_acc` improved). Kaggle's GPU quota has been far more
reliable here. This is the same split already proven for Stage 2 ->
`02b`: do the CPU-only data prep (remap + validate), then train on a
GPU elsewhere.

Trains on the **full current `merged_labels.csv`** -- image + DFDC
video + Playground synthetic + DeepDetect-2025 + DeepfakeArt -- not
just Stage 1 + Stage 2. Kaggle has no Drive access, so none of these
rows' paths (mostly absolute, from sessions that no longer exist) are
directly usable here; this notebook rebuilds a filename -> local-path
index from every attached input and remaps every row by filename
alone before training.

**Needs these attached as Kaggle inputs** ("Add Input"):
1. Stage 1's unified image dataset (`labels.csv` + `images/` + `masks/`)
   -- **re-export this from Drive after Stage 4b has run**, since
   DeepfakeArt's images/masks got copied into those same shared
   folders and need to be included.
2. Stage 2's extracted DFDC frames -- the same dataset/notebook-output
   already used for `02b`.
3. A small "extras" dataset: download `merged_labels.csv` and
   `deepdetect2025_bundle.zip` from Drive and attach them together.
4. (Optional, for warm-starting) `image_model.pth` and/or
   `video_model.pth` -- reuse whichever input already has them, or
   attach a dedicated small checkpoint dataset.

In [ ]:
import sys, os

IS_COLAB = os.path.exists("/var/colab/hostname")
IS_KAGGLE = (not IS_COLAB) and (os.path.exists("/kaggle/input") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ)
PROJECT_ROOT = "/kaggle/working/ForensicAI_Lab" if IS_KAGGLE else "/content/ForensicAI_Lab"

if not os.path.exists(PROJECT_ROOT):
    !git clone https://github.com/MRoy03/Forensic_LAB.git {PROJECT_ROOT}

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from notebooks.utils.colab_common import mount_drive, install_requirements, is_colab

if is_colab():
    mount_drive()
    install_requirements(PROJECT_ROOT)
elif IS_KAGGLE:
    !pip install -q -r requirements-train.txt
else:
    print("Not in Colab/Kaggle -- assuming requirements-train.txt is already installed locally.")

In [ ]:
import torch, cv2, numpy as np, pandas as pd, os, shutil, zipfile
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

from config.loader import get_settings, resolve_path
from services.preprocessing.pipeline import preprocess_image
from services.detection.model import ForensicMultiHeadNet

settings = get_settings()
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training on:", device)
if device == "cpu":
    print("[WARNING] No GPU detected -- set this notebook's accelerator to GPU, "
          "or this defeats the entire point of moving here from Colab.")


def find_file(root, name):
    for dirpath, _, filenames in os.walk(root, followlinks=True):
        if name in filenames:
            return Path(dirpath) / name
    return None


def find_files_by_suffix(root, suffix):
    matches = []
    for dirpath, _, filenames in os.walk(root, followlinks=True):
        for f in filenames:
            if f.lower().endswith(suffix):
                matches.append(Path(dirpath) / f)
    return matches

## 1. Locate Stage 1's unified image dataset

Same pull-in as Stage 1's own Kaggle notebook -- finds `labels.csv`
under the attached input and copies it, `images/`, and `masks/` into
this session's local disk.

In [ ]:
IMAGE_DATASET_SLUG = "forensicai-lab-image-data"  # <-- change to your attached dataset's folder name
image_dataset_root = Path("/kaggle/input") / IMAGE_DATASET_SLUG

local_images_dir = resolve_path(settings.data.images_dir)
local_masks_dir = resolve_path(settings.data.masks_dir)
local_labels_csv = resolve_path(settings.data.labels_csv)

found_labels = find_file(image_dataset_root, Path(settings.data.labels_csv).name)
assert found_labels, f"No {settings.data.labels_csv} found under {image_dataset_root} -- check IMAGE_DATASET_SLUG"
unified_root = found_labels.parent
print("Found unified image dataset at:", unified_root)

local_labels_csv.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(found_labels, local_labels_csv)
for name, dst in ((settings.data.images_dir, local_images_dir), (settings.data.masks_dir, local_masks_dir)):
    src = unified_root / name
    if src.exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)
print(f"Copied labels.csv/images/masks to local disk")

## 2. Locate and remap Stage 2's extracted DFDC frames

Same approach as `02b`: the manifest's `filename` column stores
absolute paths from the original extraction session, which won't
exist here -- find wherever the manifest and frame images actually
landed in the attached dataset, and rewrite the paths to match.

In [ ]:
FRAME_DATASET_SLUG = "dfdc-extracted-frames"  # <-- change to your actual dataset's input folder name
frame_dataset_root = Path("/kaggle/input") / FRAME_DATASET_SLUG

manifest_candidates = find_files_by_suffix(frame_dataset_root, "dfdc_frame_manifest.csv")
assert manifest_candidates, f"No dfdc_frame_manifest.csv found under {frame_dataset_root} -- check FRAME_DATASET_SLUG"
old_manifest_path = manifest_candidates[0]

jpg_candidates = find_files_by_suffix(frame_dataset_root, ".jpg")
assert jpg_candidates, f"No .jpg frames found under {frame_dataset_root}"
actual_frame_dir = jpg_candidates[0].parent
print(f"Found manifest at {old_manifest_path}, {len(jpg_candidates)} frames under {actual_frame_dir}")

frame_labels = pd.read_csv(old_manifest_path)
frame_labels["filename"] = frame_labels["filename"].apply(lambda p: str(actual_frame_dir / Path(p).name))
print(f"Remapped {len(frame_labels)} frame rows")

## 3. Locate and extract the DeepDetect-2025 bundle + the real merged_labels.csv

Download two files from Drive and attach them together as one small
Kaggle Dataset: `merged_labels.csv` (Stage 3/4/4b's current combined
manifest -- ~162k rows at last count) and `deepdetect2025_bundle.zip`
(produced by `04b`'s bundling cell). This is what actually lets this
notebook train on the FULL merged dataset -- image + DFDC + Playground
synthetic + DeepDetect-2025 + DeepfakeArt -- not just Stage 1 + 2.

DeepfakeArt's images/masks don't need a separate input: they already
live in the shared `images/`/`masks/` folders on Drive, so as long as
you re-export Stage 1's unified dataset (input 1 above) *after*
Stage 4b ran, they're already included in `local_images_dir`/
`local_masks_dir` from step 1.

In [ ]:
EXTRAS_SLUG = "forensicai-lab-extras"  # <-- change to your attached dataset's folder name
extras_root = Path("/kaggle/input") / EXTRAS_SLUG

found_merged_csv = find_file(extras_root, "merged_labels.csv")
assert found_merged_csv, f"No merged_labels.csv found under {extras_root} -- check EXTRAS_SLUG"
print("Found merged_labels.csv at:", found_merged_csv)

# Kaggle auto-extracts an uploaded zip when it builds a dataset, so
# what's actually mounted here may be the bundle's EXTRACTED contents
# (the renamed jpgs + deepdetect_manifest.csv) rather than the .zip
# file itself -- check for the already-extracted manifest first.
found_deepdetect_manifest = find_file(extras_root, "deepdetect_manifest.csv")
if found_deepdetect_manifest:
    extracted_deepdetect_dir = found_deepdetect_manifest.parent
    print(f"Found already-extracted DeepDetect-2025 bundle at {extracted_deepdetect_dir}")
else:
    deepdetect_bundle_zip = find_file(extras_root, "deepdetect2025_bundle.zip")
    extracted_deepdetect_dir = resolve_path("data/deepdetect2025_extracted")
    if deepdetect_bundle_zip:
        extracted_deepdetect_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(deepdetect_bundle_zip) as zf:
            zf.extractall(extracted_deepdetect_dir)
        print(f"Extracted DeepDetect-2025 bundle -> {extracted_deepdetect_dir}")
    else:
        extracted_deepdetect_dir = None
        print(f"No deepdetect2025_bundle.zip or deepdetect_manifest.csv found under "
              f"{extras_root} -- DeepDetect-2025 rows will fail validation below and "
              "get dropped, which is fine if you didn't mean to include them yet.")

## 4. Optional: pull in a checkpoint to warm-start from

Checks the same two input slugs above (either dataset may already
have a checkpoint bundled in) plus an optional dedicated one. Missing
this is not fatal -- training proceeds from pretrained ImageNet
weights either way if nothing is found.

In [ ]:
CHECKPOINT_SLUGS = [IMAGE_DATASET_SLUG, FRAME_DATASET_SLUG, EXTRAS_SLUG]  # <-- add/remove as needed

for ckpt_name in ("image_model.pth", "video_model.pth"):
    dst = resolve_path(f"models/{ckpt_name}")
    if dst.exists():
        print(f"{ckpt_name} already present locally")
        continue
    found = None
    for slug in CHECKPOINT_SLUGS:
        found = find_file(Path("/kaggle/input") / slug, ckpt_name)
        if found:
            break
    if found:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(found, dst)
        print(f"Copied {ckpt_name} from {found}")
    else:
        print(f"No {ckpt_name} found under any of {CHECKPOINT_SLUGS} -- "
              "proceeding without it (fine, just a missed warm-start).")

## Remap the real merged_labels.csv to this session's local paths

`merged_labels.csv` mixes several kinds of path depending on which
notebook added each row: relative (Stage 1 images), absolute paths
from a now-gone Colab session (DFDC, DeepDetect-2025), and absolute
Drive paths that don't exist on Kaggle at all (DeepfakeArt). Rather
than special-case each source, this builds one filename -> local-path
index from everything pulled in above and remaps every row by
filename alone -- the same trick `02b` already uses for DFDC frames,
generalized to all sources at once. Every filename across every
source is already unique (each notebook's own naming convention
guarantees that), so a flat lookup is safe.

In [ ]:
def build_filename_index(root):
    index = {}
    if root and Path(root).exists():
        for dirpath, _, filenames in os.walk(root, followlinks=True):
            for f in filenames:
                index[f] = str(Path(dirpath) / f)
    return index


filename_index = {}
for root in (local_images_dir, local_masks_dir, actual_frame_dir, extracted_deepdetect_dir):
    filename_index.update(build_filename_index(root))
print(f"Indexed {len(filename_index)} files across all attached inputs")

merged = pd.read_csv(found_merged_csv)
merged["filename"] = merged["filename"].apply(
    lambda f: filename_index.get(Path(f).name, f)
)

merged_path = resolve_path("data/merged_labels.csv")
merged.to_csv(merged_path, index=False)
print(f"Remapped and saved {len(merged)} rows -> {merged_path}")
print(merged["label"].value_counts())

## Validate the merged dataset before training

Same lesson as every other training notebook: catch unreadable files
up front instead of crashing mid-epoch.

In [ ]:
images_dir = local_images_dir
masks_dir = local_masks_dir

_check_df = pd.read_csv(merged_path)
_bad_indices = []

for i, row in _check_df.iterrows():
    if i > 0 and i % 2000 == 0:
        print(f"  ...checked {i}/{len(_check_df)}")
    p = Path(row["filename"])
    img_path = p if p.is_absolute() else images_dir / p
    if cv2.imread(str(img_path)) is None:
        _bad_indices.append(i)

print(f"{len(_bad_indices)} of {len(_check_df)} rows failed to load.")
if _bad_indices:
    _check_df = _check_df.drop(index=_bad_indices).reset_index(drop=True)
    _check_df.to_csv(merged_path, index=False)
    print(f"Removed {len(_bad_indices)} bad rows -- merged_labels.csv now has {len(_check_df)} rows.")

## Retrain on the merged dataset

Same fixes as Stage 3's own notebook: training-only augmentation,
sqrt-weighted oversampling (not full inverse frequency -- that caused
real_acc to collapse in an earlier 02b attempt), plain unweighted loss
(combining with class-weighting double-corrected in practice), and
best-BALANCED-ACCURACY checkpointing (not best val_loss -- on data
this imbalanced, val_loss is majority-class-dominated and doesn't
reliably track real/fake decision quality).

In [ ]:
WARM_START_FROM = "video"  # "video" or "image"
VAL_FRACTION = 0.15
NUM_EPOCHS = 5

In [ ]:
import random


class ForensicImageDataset(Dataset):
    def __init__(self, labels_csv, images_dir, masks_dir, image_size, augment=False):
        self.df = pd.read_csv(labels_csv)
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.image_size = image_size
        self.augment = augment
        self.label_to_idx = {l: i for i, l in enumerate(settings.labels.binary)}
        manip_classes = settings.labels.manipulation_types
        self.manip_to_idx = {m: i for i, m in enumerate(manip_classes)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        p = Path(row["filename"])
        img_path = p if p.is_absolute() else self.images_dir / p
        image_bgr = cv2.imread(str(img_path))
        if image_bgr is None:
            print(f"[WARN] Could not read {img_path}, substituting another sample")
            return self[(idx + 1) % len(self)]

        if self.augment:
            if random.random() < 0.5:
                image_bgr = cv2.flip(image_bgr, 1)
            alpha = random.uniform(0.85, 1.15)
            beta = random.uniform(-20, 20)
            image_bgr = cv2.convertScaleAbs(image_bgr, alpha=alpha, beta=beta)

        x = preprocess_image(image_bgr, tuple(self.image_size))

        label_idx = self.label_to_idx[row["label"]]
        row_type = row["type"]
        has_manip = 1.0 if row_type in self.manip_to_idx else 0.0
        manip_idx = self.manip_to_idx.get(row_type, 0)

        mask_path = self.masks_dir / img_path.name
        if mask_path.exists():
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            mask = cv2.resize(mask, tuple(self.image_size))
            mask = (mask > 127).astype(np.float32)[np.newaxis, :, :]
            has_mask = 1.0
        else:
            mask = np.zeros((1, *self.image_size), dtype=np.float32)
            has_mask = 0.0

        return (
            torch.from_numpy(x),
            torch.tensor(label_idx, dtype=torch.long),
            torch.tensor(manip_idx, dtype=torch.long),
            torch.from_numpy(mask),
            torch.tensor(has_mask, dtype=torch.float32),
            torch.tensor(has_manip, dtype=torch.float32),
        )

In [ ]:
manifest_length = len(pd.read_csv(merged_path))
val_size = int(manifest_length * VAL_FRACTION)
train_size = manifest_length - val_size
generator = torch.Generator().manual_seed(42)
perm = torch.randperm(manifest_length, generator=generator).tolist()
train_indices, val_indices = perm[:train_size], perm[train_size:]

train_dataset_full = ForensicImageDataset(merged_path, images_dir, masks_dir, settings.inference.image_size, augment=True)
val_dataset_full = ForensicImageDataset(merged_path, images_dir, masks_dir, settings.inference.image_size, augment=False)
train_dataset = torch.utils.data.Subset(train_dataset_full, train_indices)
val_dataset = torch.utils.data.Subset(val_dataset_full, val_indices)

train_labels = train_dataset_full.df.iloc[train_indices]["label"]
train_label_counts = train_labels.value_counts()
sample_weights = train_labels.map(lambda l: 1.0 / (train_label_counts[l] ** 0.5)).values
sampler = torch.utils.data.WeightedRandomSampler(
    weights=sample_weights, num_samples=len(sample_weights), replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
print(f"Train: {len(train_dataset)} rows, Val: {len(val_dataset)} rows")

In [ ]:
model = ForensicMultiHeadNet(
    backbone=settings.models.backbone,
    in_chans=settings.inference.input_channels,
    num_manip_classes=len(settings.labels.manipulation_types),
    pretrained=settings.models.pretrained_encoder,
).to(device)

warm_start_path = resolve_path(f"models/{'video_model.pth' if WARM_START_FROM == 'video' else 'image_model.pth'}")
if warm_start_path.exists():
    model.load_state_dict(torch.load(warm_start_path, map_location=device))
    print("Warm-started from", warm_start_path)
else:
    print(f"{warm_start_path} not found -- training from randomly-initialized weights.")

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
cls_loss_fn = torch.nn.CrossEntropyLoss()
manip_loss_fn = torch.nn.CrossEntropyLoss(reduction="none")
mask_loss_fn = torch.nn.BCEWithLogitsLoss(reduction="none")
checkpoint_path = resolve_path(settings.models.image_model_checkpoint)
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
def compute_loss(out, label_idx, manip_idx, mask, has_mask, has_manip):
    loss_cls = cls_loss_fn(out["cls_logits"], label_idx)
    manip_loss_per_sample = manip_loss_fn(out["manip_logits"], manip_idx)
    loss_manip = (manip_loss_per_sample * has_manip).sum() / (has_manip.sum() + 1e-8)
    mask_loss_per_px = mask_loss_fn(out["mask_logits"], mask)
    loss_mask = (mask_loss_per_px.mean(dim=[1, 2, 3]) * has_mask).sum() / (has_mask.sum() + 1e-8)
    return loss_cls + loss_manip + loss_mask


@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    real_idx = settings.labels.binary.index("real")
    real_correct = real_total = fake_correct = fake_total = 0

    for x, label_idx, manip_idx, mask, has_mask, has_manip in loader:
        x, label_idx, manip_idx = x.to(device), label_idx.to(device), manip_idx.to(device)
        mask, has_mask, has_manip = mask.to(device), has_mask.to(device), has_manip.to(device)
        out = model(x)
        total_loss += compute_loss(out, label_idx, manip_idx, mask, has_mask, has_manip).item()

        preds = out["cls_logits"].argmax(dim=1)
        correct_mask = preds == label_idx
        correct += correct_mask.sum().item()
        total += label_idx.size(0)

        is_real = label_idx == real_idx
        real_total += is_real.sum().item()
        real_correct += (correct_mask & is_real).sum().item()
        fake_total += (~is_real).sum().item()
        fake_correct += (correct_mask & ~is_real).sum().item()

    model.train()
    real_acc = real_correct / real_total if real_total else float("nan")
    fake_acc = fake_correct / fake_total if fake_total else float("nan")
    balanced_acc = (real_acc + fake_acc) / 2
    return total_loss / len(loader), correct / total, real_acc, fake_acc, balanced_acc


# Saves after every epoch when balanced_acc improves, AND unconditionally
# after every epoch regardless -- so a GPU disconnect mid-run only costs
# the current epoch, not the whole session (the exact failure that
# prompted moving this notebook to Kaggle in the first place).
best_balanced_acc = -1.0
model.train()
for epoch in range(NUM_EPOCHS):
    running_loss = 0.0
    for x, label_idx, manip_idx, mask, has_mask, has_manip in train_loader:
        x, label_idx, manip_idx = x.to(device), label_idx.to(device), manip_idx.to(device)
        mask, has_mask, has_manip = mask.to(device), has_mask.to(device), has_manip.to(device)

        optimizer.zero_grad()
        out = model(x)
        loss = compute_loss(out, label_idx, manip_idx, mask, has_mask, has_manip)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    val_loss, val_acc, real_acc, fake_acc, balanced_acc = evaluate(val_loader)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} -- train_loss: {train_loss:.4f}  val_loss: {val_loss:.4f}  "
          f"val_acc: {val_acc:.2%}  real_acc: {real_acc:.2%}  fake_acc: {fake_acc:.2%}  "
          f"balanced_acc: {balanced_acc:.2%}")

    torch.save(model.state_dict(), resolve_path("models/image_model_last_epoch.pth"))
    if balanced_acc > best_balanced_acc:
        best_balanced_acc = balanced_acc
        torch.save(model.state_dict(), checkpoint_path)
        print(f"  -> new best balanced_acc ({balanced_acc:.2%}), saved checkpoint to {checkpoint_path}")

print(f"\nBest balanced_acc: {best_balanced_acc:.2%}")

## Getting the checkpoint out of Kaggle

Download `models/image_model.pth` (the best-balanced-accuracy
checkpoint) from this run's Output tab and upload it to
`ForensicAI_Lab/models/image_model.pth` on your Drive -- this is now
the final checkpoint, trained on the full merged dataset. Run
`06_stage6_final_export_and_onnx.ipynb` next to export it.